# Modelado 70 / 30

* Ines Gonzales
* Jose Arenas
* Jhonier Correa
  
En este punto los datos se encuentran imputados y smoteados.

# Librerias

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import metrics

# Modelos para evaluación
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


# Lectura de datos y correccion de tipos

In [5]:
data = pd.read_csv(r'datos\datos_smoteados.csv', index_col=0)
print(data.shape)
data.head()

(9688, 40)


,payer_type,patient_age,breast_cancer_diagnosis_code,population,density,age_median,age_60s,age_70s,married,never_married,...,race_white,race_other,race_multiple,hispanic,disabled,poverty,limited_english,veteran,Ozone,tipo_metastasis
0,COMMERCIAL,62,C50.2_C50.4,39121.878790,2295.939394,38.200000,10.248485,5.951515,50.245455,35.290909,...,54.030303,11.645455,10.081818,37.948485,8.957576,10.109091,8.057576,4.103030,42.301121,Linfatico
1,COMMERCIAL,43,C50.1,21996.683330,626.236667,37.906667,11.426667,6.565000,55.753333,27.195000,...,75.820000,3.816667,6.898333,19.370000,11.253333,9.663333,3.356667,7.446667,40.108207,Linfatico
2,COMMERCIAL,45,C50.2_C50.4,32795.325580,1896.220930,42.871429,12.280952,8.216667,52.604762,31.142857,...,65.014286,5.428571,8.611905,16.716667,8.845238,8.688095,5.280952,4.809524,42.070075,Linfatico
3,'MEDICARE ADVANTAGE',66,C50.9_C79.81,5643.771429,219.362857,45.180000,15.685714,9.445714,51.322857,30.831429,...,93.528571,1.000000,3.262857,2.465714,13.717143,8.888235,0.638235,7.745714,40.107248,VisceralSistemico
4,COMMERCIAL,60,C50.3_C50.5,3404.100000,25.733333,42.790000,14.043333,8.526667,55.886667,24.526667,...,96.406667,0.406667,1.913333,2.723333,15.260000,10.890000,0.503333,7.506667,37.646770,Linfatico


In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 9688 entries, 0 to 9687
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   payer_type                    9688 non-null   str    
 1   patient_age                   9688 non-null   int64  
 2   breast_cancer_diagnosis_code  9688 non-null   str    
 3   population                    9688 non-null   float64
 4   density                       9688 non-null   float64
 5   age_median                    9688 non-null   float64
 6   age_60s                       9688 non-null   float64
 7   age_70s                       9688 non-null   float64
 8   married                       9688 non-null   float64
 9   never_married                 9688 non-null   float64
 10  family_size                   9688 non-null   float64
 11  income_household_median       9688 non-null   float64
 12  income_household_10_to_15     9688 non-null   float64
 13  income_househo

In [7]:
data['payer_type'] = data['payer_type'].astype('category')
data['breast_cancer_diagnosis_code'] = data['breast_cancer_diagnosis_code'].astype('category')
data['tipo_metastasis'] = data['tipo_metastasis'].astype('category')

# Division y creacion de pipelines

In [ ]:
# 1. Separar características (X) y objetivo (y)
X = data.drop(columns=['tipo_metastasis'])
y = data['tipo_metastasis']

# Codificar la variable objetivo (y) si es categórica
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Pipeline para variables numéricas (escalado)
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

# Pipeline para variables categóricas (imputación + One-Hot Encoding)
categorical_transformer = Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

# Preprocesador global combinando ambos tipos de columnas
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print(f'Datos de entreamiento {X_train.shape[0]}')
print(f'Datos de evaluacion {X_test.shape[0]}')

Datos de entreamiento 7750
Datos de evaluacion 1938


array([1, 1, 1, ..., 0, 0, 0], shape=(9688,))

# Modelado

In [ ]:
# Diccionario con los modelos a comparar
modelos = {
    'Red Neuronal': MLPClassifier(
    hidden_layer_sizes=(10),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42,
    early_stopping=True),

    'k-NN': KNeighborsClassifier(n_neighbors=4),

    'SVM': SVC(kernel='rbf', C=1, class_weight='balanced', random_state=42),

    'XGBoost': XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
    ),

    'Random Forest': RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    criterion='gini',
    class_weight='balanced',
    random_state=42)
}


# DataFrame para almacenar los resultados comparativos de clasificación
medidas = pd.DataFrame(index=[
    'Accuracy', 
    'Precision (Macro)', 
    'Recall (Macro)', 
    'F1-Score (Macro)', 

    #'ROC-AUC (OvR Macro)', 
    
    'Parámetros Clave'
])

# 3. Entrenamiento y evaluación
for nombre_modelo, algoritmo in modelos.items():
    pipeline_completo = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('modelo', algoritmo)
    ])
    
    # Entrenar pipeline
    pipeline_completo.fit(X_train, y_train)
    
    # Predicción de clases y de probabilidades (necesarias para ROC)
    y_pred = pipeline_completo.predict(X_test)

    #y_proba = pipeline_completo.predict_proba(X_test)
    
    # Métrica ROC-AUC para 3 clases (One-vs-Rest)

    #roc_auc = metrics.roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    
    '''# Impresión detallada del informe y ROC-AUC
    print(f"{'='*20} INFORME DE CLASIFICACIÓN: {nombre_modelo} {'='*20}")
    print(metrics.classification_report(y_true=y_test, y_pred=y_pred, target_names=le.classes_))
    print(f"ROC-AUC Score (OvR Macro): {roc_auc:.4f}\n")'''
    
    # Métricas individuales
    acc = metrics.accuracy_score(y_test, y_pred)
    prec = metrics.precision_score(y_test, y_pred, average='macro')
    rec = metrics.recall_score(y_test, y_pred, average='macro')
    f1 = metrics.f1_score(y_test, y_pred, average='macro')
    
    params = str(algoritmo.get_params())
    
    # Almacenar métricas en el DataFrame
    medidas[nombre_modelo] = [
        round(acc, 4),
        round(prec, 4),
        round(rec, 4),
        round(f1, 4),

        #round(roc_auc, 4),

        params[:60] + '...'
    ]

# Medidas de evaluacion

In [11]:
medidas

,Red Neuronal,k-NN,SVM,XGBoost,Random Forest
Accuracy,0.6032,0.5413,0.4479,0.5846,0.4757
Precision (Macro),0.5632,0.4133,0.4113,0.4465,0.432
Recall (Macro),0.3328,0.4073,0.4136,0.3999,0.448
F1-Score (Macro),0.2544,0.4022,0.3955,0.3991,0.4255
Parámetros Clave,"{'activation': 'relu', 'alpha': 0.0001, 'batch...","{'algorithm': 'auto', 'leaf_size': 30, 'metric...","{'C': 1, 'break_ties': False, 'cache_size': 20...","{'objective': 'multi:softprob', 'base_score': ...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."
